# Assignment APIs tutorial

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

Required files:
- Directory called "millionaire_client"
- 1 colab notebook

Both files must be saved in a directory in your Google Drive, for example:
```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```

### Sign up procedure
Before showing you how the api work, you need to signup from a web browser.
- Paste this link into your browser [http://131.175.15.22:51111/](http://131.175.15.22:51111/) this is where the demo is deployed
- You will see a standard login/sign up screen, please click on sign up
- In the "email" field please enter your politecnico email, you are allwed to create only 1 account using the same email you registered to the NLP course
- Choose whever username/password you prefer (be creative ;))

### Game interaction

Once you signed up, you can start interacting already from the api.

First of all, let's connect your drive to this Colab Notebook

In [3]:
from google.colab import drive
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TextStreamer
import os

drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [31]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)



Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [32]:
def generate_outputs(system_message, user_message, stream=False):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    outputs = pipe(
        messages,
        max_new_tokens=10,
        do_sample=False,
        temperature=None,
        top_p=None,
        return_full_text=False
    )
    response = outputs[0]['generated_text'][-1]['content']
    return response



Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [33]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP/project_modules', '/content/gdrive/MyDrive/NLP/millionaire_client', '/content/gdrive/MyDrive/NLP']


Let's import the client classes

In [34]:
from millionaire_client import MillionaireClient, AuthenticationError

Please verify that you see a directory named `millionaire_client/` directly under the `project_modules/` directory in the output above. If it's not there, you need to ensure the `millionaire_client` folder (containing `__init__.py` and other client files) is placed correctly inside the `project_modules` directory in your Google Drive, following the structure `gDrive_home/Colab Notebooks/NLP_assignment/millionaire_client/` relative to your mounted drive path.

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [35]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [36]:
API_URL = "http://131.175.15.22:51111/"
username = "riccardo"
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [37]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, riccardo! (Role: student)


After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [38]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [39]:
# Choose a competition ID
comp_id = 1

After choosing a competition, we can start a game! We can choose to start a game by calling `game = client.game.start(competition_id=comp_id)`. The object game is the one that is handling the game itself, we can call:
- game.current_question.text : to know the current question in text format
- game.current_level: to check the current level of difficulty of the question
- game.current_question.options: to check the possible choices we have to answer the question
- game.answer: to send to the server the answer we choose (the integer corresponding to our choice) and get the response (either correct or incorrect)

WATCH OUT! Each question has a timer, you have maximum 30 seconds to answer the question. As of now, if you exceed the maximum allowed time, there is not a "push notification". You still have to submit your answer anyway and, even though the answer was correct, you will get a TimedOut response!

In [40]:
import re

def extract_answer_id(output, num_options):
    """Estrae il numero di risposta (0-based) dall'output del modello."""
    # cerca il primo numero valido nell'output
    matches = re.findall(r'\b([0-9])\b', output)
    for m in matches:
        if int(m) < num_options:
            return int(m)
    return 0  # fallback

def play_game(game):
    while game.in_progress:
        print("🔄 Loop iteration started")

        question = game.current_question
        print(f"📋 Question object: {question}")

        if not question:
            print("No question available. Game may have ended.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}")

        options_text = "\n".join([f"[{opt.id}] {opt.text}" for opt in question.options])
        print(f"Options: {options_text}")

        time_left = game.time_remaining
        print(f"⏱️ Time remaining: {time_left}")

        print("🤖 Calling model...")  # se si blocca QUI → problema nel modello
        output = generate_outputs(
            system_message=(
                "You are a quiz expert. Answer the multiple choice question.\n"
                "Reply with ONLY the number of the correct option (0, 1, 2, or 3). "
                "No explanation, just the number."
            ),
            user_message=(
                f"Question: {question.text}\n\n"
                f"Options:\n{options_text}\n\n"
                "Answer (number only):"
            ),
        )
        print(f"✅ Model output: '{output}'")  # se si blocca PRIMA → mai stampato

        answer_id = extract_answer_id(output, len(question.options))
        print(f"📤 Submitting answer: {answer_id}")

        result = game.answer(answer_id)
        print(f"📥 Result received: correct={result.correct}, timed_out={result.timed_out}")

        # resto del codice...

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [41]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Session ID: 37239
Total number of questions: 15

🔄 Loop iteration started
📋 Question object: Question(id=1072, text='How did the Neo-Assyrian Empire ensure the loyalty of its conquered territories?', options=[Option(id=0, text='By relocating entire populations to new regions'), Option(id=1, text='By destroying local cultures and identities'), Option(id=2, text='By imposing a common language and religion'), Option(id=3, text='By allowing local rulers to maintain power as vassals')], level=0)

--- Level 1 ---
Q: How did the Neo-Assyrian Empire ensure the loyalty of its conquered territories?
Options: [0] By relocating entire populations to new regions
[1] By destroying local cultures and identities
[2] By imposing a common language and religion
[3] By allowing local rulers to maintain power as vassals
⏱️ Time remaining: 29.929151
🤖 Calling model...


KeyboardInterrupt: 

In [ ]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. mark: $200.00 (Level 2)
  2. nicolo: $100.00 (Level 1) <-- YOU
  3. ftoschi: $100.00 (Level 1)
